In [1]:
from unc_handling import UG_prompter
from DataLoader import DataLoader
from segmentation import Segmentation
from segmentation_util import combine_prompt_sets
from evaluation import Evaluator, compare_to_recontours, save_evaluation_results
from pathlib import Path

root = r"C:\Users\20202310\Desktop\MSc scriptie\MSc_Graduation_Project\data\LUNDPROBE\ExtendedSamples\development"

methods_available = ["raycast", "local_normals"]
propagation_styles = ['default', 'full', 'prompt_based', 'central_start', 'central_partitions']
method = methods_available[1]
propagation_style = propagation_styles[3]

rootpath = Path(root)
subjects = sorted([p.name for p in rootpath.iterdir() if p.is_dir()])
print(subjects)

nn_Unet_Evals = []
SAM_Evals = []
Recontour_evals = []


['newAcq_050f229dc2bdb64c', 'newAcq_0b4940fa31a1d650', 'newAcq_0cc559a8bd82a14a', 'newAcq_1b911d6cb2348f30', 'newAcq_1e0f8b9b01ce5f0b', 'newAcq_250d6075dd465a1a', 'newAcq_433a8d44fddd5b7f', 'newAcq_47ceabdbca398517', 'newAcq_486b7494ee9d71e7', 'newAcq_4a136e8fe320bd13']


In [2]:
def pipeline(target_mm,method,propagation_style,unc_band_thr_mm,pixel_interval,angle_step,weighting_list,results_filename):
    
    for subject_nr in range(len(subjects)):
        data = DataLoader(parentfolder=root,subject_nr=subject_nr,volume_of_interest="CTVT",verbose=True)
        unc_handler = UG_prompter(data=data)
        seg_handler = Segmentation(data=data)

        unc_handler.threshold_uncertainty_map(unc_threshold=None, target_mm=target_mm, method="raycast", mode="median") #unc_threshold=0.033470
        unc_handler.compute_band_thickness(method=method)

        nietjes_prompts = unc_handler.generate_prompts_nietjes(unc_band_thr_mm=unc_band_thr_mm,
        interpix_dist=3,
        pixel_interval=pixel_interval,
        angle_step=angle_step,
        method=method)

        bbox_prompts = unc_handler.generate_prompts_boxes(band_threshold=0.0)

        dense_prompt = seg_handler.load_dense_prompt()
        dense_and_nietjes_prompts = combine_prompt_sets(prompt_dict_list = [dense_prompt, nietjes_prompts])

        prompt_sets = [dense_and_nietjes_prompts, bbox_prompts]
        prompt_names = ["Dense_and_nietjes", "Uncertainty_bboxes"]
        
        prompt_weights = weighting_list

        seg_handler.compile_prompt_sets(prompt_dict_list=prompt_sets, prompt_set_names=prompt_names, prompt_set_weights=prompt_weights)

        seg_handler.run_segmentation_sets(propagation_style=propagation_style, weighting_strategy="custom", threshold=0.0)
        seg_handler.remove_distant_slices(tolerance_frames=0)

        eval_handler = Evaluator(segmentation=seg_handler)
        metrics = eval_handler.compute_all(surface_dice_tol=1.0)
        SAM_Evals.append(metrics)

        save_evaluation_results(SAM_Evals, "results", filename=results_filename)

In [3]:
#Default settings:
#pipeline(target_mm=3.0,method=method,propagation_style=propagation_style,unc_band_thr_mm=4.0,pixel_interval=10,weighting_list=[0.2, 0.8],results_filename="promptset_densenietjes_bbox.csv")

In [4]:
from pathlib import Path

output_folder = Path("promptset_gridsearch_results")
output_folder.mkdir(parents=True, exist_ok=True)

parameter_log_path = output_folder / "run_parameters.txt"

run_nr = 1

parameter_lists = [
    {"target_mm": 2.0, "method": "local_normals", "unc_band_thr_mm": 3.0, "pixel_interval": 10, "angle_step": 5, "weighting_list": [0.2, 0.8], "propagation_style": propagation_style},
    {"target_mm": 3.0, "method": "local_normals", "unc_band_thr_mm": 3.0, "pixel_interval": 10, "angle_step": 5, "weighting_list": [0.2, 0.8], "propagation_style": propagation_style},
    {"target_mm": 4.0, "method": "local_normals", "unc_band_thr_mm": 3.0, "pixel_interval": 10, "angle_step": 5, "weighting_list": [0.2, 0.8], "propagation_style": propagation_style},
    {"target_mm": 5.0, "method": "local_normals", "unc_band_thr_mm": 3.0, "pixel_interval": 10, "angle_step": 5, "weighting_list": [0.2, 0.8], "propagation_style": propagation_style},

    {"target_mm": 3.0, "method": "local_normals", "unc_band_thr_mm": 2.0, "pixel_interval": 10, "angle_step": 5, "weighting_list": [0.2, 0.8], "propagation_style": propagation_style},
    {"target_mm": 3.0, "method": "local_normals", "unc_band_thr_mm": 4.0, "pixel_interval": 10, "angle_step": 5, "weighting_list": [0.2, 0.8], "propagation_style": propagation_style},
    {"target_mm": 3.0, "method": "local_normals", "unc_band_thr_mm": 5.0, "pixel_interval": 10, "angle_step": 5, "weighting_list": [0.2, 0.8], "propagation_style": propagation_style},

    {"target_mm": 3.0, "method": "local_normals", "unc_band_thr_mm": 3.0, "pixel_interval": 5, "angle_step": 2.5, "weighting_list": [0.2, 0.8], "propagation_style": propagation_style},
    {"target_mm": 3.0, "method": "local_normals", "unc_band_thr_mm": 3.0, "pixel_interval": 1, "angle_step": 1, "weighting_list": [0.2, 0.8], "propagation_style": propagation_style},

    {"target_mm": 3.0, "method": "local_normals", "unc_band_thr_mm": 3.0, "pixel_interval": 10, "angle_step": 5, "weighting_list": [0.5, 0.5], "propagation_style": propagation_style},
    {"target_mm": 3.0, "method": "local_normals", "unc_band_thr_mm": 3.0, "pixel_interval": 10, "angle_step": 5, "weighting_list": [0.8, 0.2], "propagation_style": propagation_style},
]


with open(parameter_log_path, "w") as f:
    f.write("Run parameter overview\n")
    f.write("=" * 80 + "\n\n")

    for params in parameter_lists:

        results_filename = output_folder / f"run{run_nr}.csv"

        f.write(f"run{run_nr}.csv\n")
        f.write(f"  target_mm: {params['target_mm']}\n")
        f.write(f"  method: {params['method']}\n")
        f.write(f"  unc_band_thr_mm: {params['unc_band_thr_mm']}\n")
        f.write(f"  pixel_interval: {params['pixel_interval']}\n")
        f.write(f"  angle_step: {params['angle_step']}\n")
        f.write(f"  weighting_list: {params['weighting_list']}\n")
        f.write(f"  propagation_style: {params['propagation_style']}\n")
        f.write("\n")

        pipeline(
            target_mm=params["target_mm"],
            method=params["method"],
            propagation_style=params["propagation_style"],
            unc_band_thr_mm=params["unc_band_thr_mm"],
            pixel_interval=params["pixel_interval"],
            angle_step=params["angle_step"],
            weighting_list=params["weighting_list"],
            results_filename=str(results_filename),
        )

        run_nr += 1

Loaded subject newAcq_050f229dc2bdb64c with volume of interest 'CTVT'
Image shape: (88, 1024, 1024), Mask shape: (88, 1024, 1024), Uncertainty map shape: (88, 1024, 1024), Ground truth shape: (88, 1024, 1024)
Image spacing (z, y, x): (2.5, 0.46880000829696655, 0.46880000829696655)
Initilialized UG_prompter for subject 0 with volume of interest 'CTVT'
Image shape: (88, 1024, 1024), Mask shape: (88, 1024, 1024), Uncertainty map shape: (88, 1024, 1024).
Initialized Segmentation for subject 0 with volume of interest 'CTVT'
Mask shape: (88, 1024, 1024)
Building SAM predictor from checkpoint: checkpoints/MedSAM2_latest.pt


c:\Users\20202310\Desktop\MSc scriptie\MSc_Graduation_Project\sam2\modeling\sam\transformer.py:23: UserWarning: Flash Attention is disabled as it requires a GPU with Ampere (8.0) CUDA capability.
  OLD_GPU, USE_FLASH_ATTN, MATH_KERNEL_ON = get_sdpa_settings()


iter=00 | thr=0.194962 | band=0.94 mm | error=1.06
iter=01 | thr=0.097481 | band=1.41 mm | error=0.59
iter=02 | thr=0.048740 | band=1.99 mm | error=0.01
iter=03 | thr=0.024370 | band=2.46 mm | error=0.46
iter=04 | thr=0.036555 | band=2.23 mm | error=0.23
iter=05 | thr=0.042648 | band=1.99 mm | error=0.01
iter=06 | thr=0.039602 | band=2.11 mm | error=0.11
iter=07 | thr=0.041125 | band=1.99 mm | error=0.01
iter=08 | thr=0.040363 | band=2.11 mm | error=0.11
iter=09 | thr=0.040744 | band=2.05 mm | error=0.05
iter=10 | thr=0.040934 | band=1.99 mm | error=0.01
iter=11 | thr=0.040839 | band=1.99 mm | error=0.01
iter=12 | thr=0.040792 | band=1.99 mm | error=0.01
iter=13 | thr=0.040768 | band=1.99 mm | error=0.01
iter=14 | thr=0.040756 | band=1.99 mm | error=0.01
iter=15 | thr=0.040750 | band=1.99 mm | error=0.01
iter=16 | thr=0.040747 | band=1.99 mm | error=0.01
iter=17 | thr=0.040745 | band=1.99 mm | error=0.01
iter=18 | thr=0.040745 | band=1.99 mm | error=0.01
iter=19 | thr=0.040744 | band=2

c:\Users\20202310\Desktop\MSc scriptie\MSc_Graduation_Project\sam2\sam2_video_predictor_npz.py:965: UserWarning: cannot import name '_C' from 'sam2' (c:\Users\20202310\Desktop\MSc scriptie\MSc_Graduation_Project\sam2\__init__.py)

Skipping the post-processing step due to the error above. You can still use SAM 2 and it's OK to ignore the error above, although some post-processing functionality may be limited (which doesn't affect the results in most cases; see https://github.com/facebookresearch/sam2/blob/main/INSTALL.md).
  pred_masks_gpu = fill_holes_in_mask_scores(


Adding prompt(s) on slice 28
Adding prompt(s) on slice 29
Adding prompt(s) on slice 30
Adding prompt(s) on slice 31
Adding prompt(s) on slice 32
Adding prompt(s) on slice 33
Adding prompt(s) on slice 34
Adding prompt(s) on slice 35
Adding prompt(s) on slice 36
Adding prompt(s) on slice 37
Adding prompt(s) on slice 38
Adding prompt(s) on slice 39
Adding prompt(s) on slice 40
Adding prompt(s) on slice 41
Adding prompt(s) on slice 42
Adding prompt(s) on slice 43
Adding prompt(s) on slice 44
Forward propagation (from middle slice 36)...


...

propagate in video: 100%|██████████| 46/46 [00:11<00:00,  3.98it/s]


Backward propagation (from middle slice 42)...


propagate in video: 100%|██████████| 43/43 [00:10<00:00,  4.23it/s]


Kept slices 33 to 51. Removed predicted segmentation outside dense mask ±0 slices.
Saved CSV to:   results\promptset_gridsearch_results\run6.csv.csv
Saved Excel to: results\promptset_gridsearch_results\run6.csv.xlsx
Loaded subject newAcq_050f229dc2bdb64c with volume of interest 'CTVT'
Image shape: (88, 1024, 1024), Mask shape: (88, 1024, 1024), Uncertainty map shape: (88, 1024, 1024), Ground truth shape: (88, 1024, 1024)
Image spacing (z, y, x): (2.5, 0.46880000829696655, 0.46880000829696655)
Initilialized UG_prompter for subject 0 with volume of interest 'CTVT'
Image shape: (88, 1024, 1024), Mask shape: (88, 1024, 1024), Uncertainty map shape: (88, 1024, 1024).
Initialized Segmentation for subject 0 with volume of interest 'CTVT'
Mask shape: (88, 1024, 1024)
Building SAM predictor from checkpoint: checkpoints/MedSAM2_latest.pt
iter=00 | thr=0.194962 | band=0.94 mm | error=2.06
iter=01 | thr=0.097481 | band=1.41 mm | error=1.59
iter=02 | thr=0.048740 | band=1.99 mm | error=1.01
iter=03

KeyboardInterrupt: 